# Geocoding Housing Projects

---

## Overview

This notebook adds latitude/longitude coordinates to housing project addresses.

**Input:** housing_projects_clean.csv

**Output:** housing_projects_with_coords.csv

**Data Source:** Alameda County Address Points (62,225 addresses)

In [ ]:
# Import libraries
import pandas as pd
import numpy as np

print('✅ Libraries imported')

## 1. Load Data

In [ ]:
# Load housing projects
df_projects = pd.read_csv('/Users/johngage/berkeley-data/housing_projects_clean.csv')
print(f'Projects: {len(df_projects):,}')

# Load Alameda County address data
df_alameda = pd.read_csv('/Users/johngage/berkeley-data/berkeley_addresses_with_fields.csv')
print(f'Alameda addresses: {len(df_alameda):,}')

## 2. Create Complete Lookup Table

Create normalized address lookup with all street name/type variations.

**Key transformations:**
- Numbered streets: FIFTH ↔ 5TH, SIXTH ↔ 6TH, etc.
- Street types: AV ↔ Ave ↔ Avenue, WY ↔ Way, ST ↔ Street, etc.
- Case variations: Upper, lower, title case

In [ ]:
def get_street_name_variations(street_name):
    """Convert between numbered and word street names"""
    conversions = {
        '1ST': 'FIRST', '2ND': 'SECOND', '3RD': 'THIRD',
        '4TH': 'FOURTH', '5TH': 'FIFTH', '6TH': 'SIXTH',
        '7TH': 'SEVENTH', '8TH': 'EIGHTH', '9TH': 'NINTH',
        '10TH': 'TENTH'
    }
    
    reverse = {v: k for k, v in conversions.items()}
    street_upper = str(street_name).upper()
    
    variations = [street_name]
    
    if street_upper in conversions:
        word = conversions[street_upper]
        variations.extend([word, word.title(), word.lower()])
    
    if street_upper in reverse:
        num = reverse[street_upper]
        variations.extend([num, num.title(), num.lower()])
    
    variations.extend([street_name.upper(), street_name.lower(), street_name.title()])
    
    return list(set(variations))

def get_type_variations(street_type):
    """Get all street type variations"""
    type_map = {
        'ST': ['ST', 'St', 'STREET', 'Street'],
        'AV': ['AV', 'Av', 'AVE', 'Ave', 'AVENUE', 'Avenue'],
        'WY': ['WY', 'Wy', 'WAY', 'Way'],
        'BL': ['BL', 'Bl', 'BLVD', 'Blvd', 'BOULEVARD', 'Boulevard'],
        'RD': ['RD', 'Rd', 'ROAD', 'Road'],
        'DR': ['DR', 'Dr', 'DRIVE', 'Drive'],
        'LN': ['LN', 'Ln', 'LANE', 'Lane'],
        'PL': ['PL', 'Pl', 'PLACE', 'Place'],
        'CT': ['CT', 'Ct', 'COURT', 'Court'],
        'SQ': ['SQ', 'Sq', 'SQUARE', 'Square'],
    }
    
    street_type_upper = str(street_type).upper()
    
    if street_type_upper in type_map:
        return type_map[street_type_upper]
    
    return [street_type, str(street_type).upper(), str(street_type).title()]

print('✅ Functions defined')

In [ ]:
# Build lookup table
print('Building complete lookup table...')

df_alameda['ST_NUM_clean'] = pd.to_numeric(df_alameda['ST_NUM'], errors='coerce')

lookup_rows = []

for idx, row in df_alameda.iterrows():
    if pd.isna(row['ST_NUM']) or pd.isna(row['FEANME']) or pd.isna(row['FEATYP']):
        continue
    
    try:
        street_num = str(int(float(row['ST_NUM'])))
    except:
        continue
    
    street_name = str(row['FEANME']).strip()
    street_type = str(row['FEATYP']).strip()
    
    name_vars = get_street_name_variations(street_name)
    type_vars = get_type_variations(street_type)
    
    for name_var in name_vars:
        for type_var in type_vars:
            normalized = f"{street_num} {name_var} {type_var}"
            
            lookup_rows.append({
                'normalized_address': normalized,
                'latitude': row['latitude'],
                'longitude': row['longitude'],
                'APN': row['APN']
            })
    
    if idx % 10000 == 0:
        print(f'  Processed {idx:,}...')

df_lookup = pd.DataFrame(lookup_rows)
df_lookup = df_lookup.drop_duplicates(subset=['normalized_address'], keep='first')

print(f'\n✅ Created lookup: {len(df_lookup):,} variations')

# Save
df_lookup.to_csv('/Users/johngage/berkeley-data/alameda_lookup_complete.csv', index=False)
print('✅ Saved lookup table')

## 3. Match Addresses

In [ ]:
# Match project addresses to coordinates
matched = 0

for idx, row in df_projects.iterrows():
    project_addr = row['address_clean']
    
    match = df_lookup[df_lookup['normalized_address'] == project_addr]
    
    if len(match) > 0:
        df_projects.at[idx, 'latitude'] = match.iloc[0]['latitude']
        df_projects.at[idx, 'longitude'] = match.iloc[0]['longitude']
        df_projects.at[idx, 'apn'] = str(match.iloc[0]['APN'])
        matched += 1

print(f'Matched: {matched}/{len(df_projects)} ({matched/len(df_projects)*100:.1f}%)')

## 4. Save with Coordinates

In [ ]:
output_file = '/Users/johngage/berkeley-data/housing_projects_with_coords.csv'
df_projects.to_csv(output_file, index=False)

print(f'✅ Saved: {output_file}')
print(f'   Projects: {len(df_projects):,}')
print(f'   With coordinates: {df_projects["latitude"].notna().sum():,}')